# Trace2Tree — Navigating Traces as a Tree Structure

The PyTorch profiler generates a JSON trace file containing events with timestamps and durations — but it lacks explicit call stack or dependency information. TraceLens parses this JSON and reconstructs the **hierarchical dependencies** from CPU ops down to GPU kernels.

This notebook demonstrates the `TreePerfAnalyzer` interface, which lets you traverse this tree structure to understand exactly how CPU operations map to GPU kernel execution.

## Load the Trace

We load the trace file using `TreePerfAnalyzer.from_file()`. Setting `add_python_func=True` includes Python-level function calls in the tree, giving us the full call stack from Python code down to GPU kernels.

In [ ]:
from TraceLens.TreePerf import TreePerfAnalyzer

trace="HPCTrainingExamples/MLExamples/PyTorch_Profiling/traces/trace_0_20_0.json"

analyzer=TreePerfAnalyzer.from_file(trace,add_python_func=True)
trace_tree=analyzer.tree
print(f"Total events in trace:{len(trace_tree.events)}")

Building tree with add_python_func=True
Building CPU op tree with add_python_func=True
Total events in trace:40533


## Analyzing a Specific Operation

Let's pick a specific operation to explore. We'll find the first `aten::convolution_backward` — one of the most expensive ops in a ResNet training workload.

In [4]:
event_interest = next(
    evt
    for evt in trace_tree.events
    if evt.get("name") == "aten::convolution_backward" and evt.get("cat") == "cpu_op"
)

print(f"Found operation: {event_interest['name']}")
print(f"Duration: {event_interest.get('dur', 0):.2f} µs")
print(f"UID: {event_interest['UID']}")

Found operation: aten::convolution_backward
Duration: 108.54 µs
UID: 12894


### Viewing the Subtree

`traverse_subtree_and_print()` shows everything below this operation in the call stack — the runtime calls, and the GPU kernels they launched. This lets you see exactly what work the GPU did for this single CPU op.

In [ ]:
print(f"Subtree for {event_interest['name']}:\n")
trace_tree.traverse_subtree_and_print(event_interest)

Subtree for aten::convolution_backward:

└── UID: 12894, Category: cpu_op, Name: aten::convolution_backward
    ├── UID: 24148, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
    │   └── UID: 24766, Category: kernel, Name: SubTensorOpWithScalar1d, Duration: 2.203
    ├── UID: 24154, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
    │   └── UID: 24768, Category: kernel, Name: igemm_bwd_gtcx3_nhwc_fp16_bx0_ex0_bt128x128x32_wt32x32x8_ws1x1_w.., Duration: 16.304
    ├── UID: 24176, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
    │   └── UID: 24770, Category: kernel, Name: SubTensorOpWithScalar1d, Duration: 2.724
    └── UID: 24180, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
        └── UID: 24772, Category: kernel, Name: igemm_wrw_gtcx3_nhwc_fp16_bx0_ex0_bt256x256x32_wt32x32x8_ws2x2_w.., Duration: 15.143


### Walking Up the Parent Chain

`traverse_parents_and_print()` walks upward from the operation to show which part of the model triggered it — e.g., which layer, which `nn.Module`, and which profiler step. Adding `cpu_op_fields` displays the operation's input shapes and types along the way.

In [ ]:

print(f"Parent chain for {event_interest['name']}:\n")
root = trace_tree.traverse_parents_and_print(
    event_interest, cpu_op_fields=("Input Dims", "Input type")
)
print(f"\nRoot event: {root['name']}")

Parent chain for aten::convolution_backward:

Node:
  UID: 12894, Category: cpu_op, Name: aten::convolution_backward
    Input Dims: [[256, 2048, 1, 1], [256, 512, 1, 1], [2048, 512, 1, 1], [], [], [], [], [], [], [], []]
    Input type: ['c10::Half', 'c10::Half', 'c10::Half', 'ScalarList', 'ScalarList', 'ScalarList', 'ScalarList', 'Scalar', 'ScalarList', 'Scalar', 'ScalarList']
1-up:
  UID: 12892, Category: cpu_op, Name: ConvolutionBackward0
    Input Dims: [[256, 2048, 1, 1]]
    Input type: ['c10::Half']
2-up:
  UID: 12891, Category: cpu_op, Name: autograd::engine::evaluate_function: ConvolutionBackward0

Root event: autograd::engine::evaluate_function: ConvolutionBackward0


### Navigating Parent and Children Programmatically

You can also navigate the tree structure in code using `get_parent_event()` and `get_children_events()`. This is useful for programmatic analysis — e.g., going to the parent event and inspecting what *other* operations it launched alongside the one we've been looking at.

In [ ]:
# Get parent using tree method
parent_evt = trace_tree.get_parent_event(event_interest)
if parent_evt:
    print(f"Parent: {parent_evt['name']} (cat: {parent_evt['cat']})\n")

# Get children using tree method
children = trace_tree.get_children_events(event_interest)
print(f"Children ({len(children)}):")
for child in children:
    print(f"  - {child['name']} (cat: {child['cat']})")

Parent: ConvolutionBackward0 (cat: cpu_op)

Children (11):
  - aten::resize_ (cat: cpu_op)
  - aten::resize_ (cat: cpu_op)
  - hipGetDevicePropertiesR0600 (cat: cuda_runtime)
  - hipExtModuleLaunchKernel (cat: cuda_runtime)
  - hipExtModuleLaunchKernel (cat: cuda_runtime)
  - aten::resize_ (cat: cpu_op)
  - aten::resize_ (cat: cpu_op)
  - aten::empty (cat: cpu_op)
  - hipGetDevicePropertiesR0600 (cat: cuda_runtime)
  - hipExtModuleLaunchKernel (cat: cuda_runtime)
  - hipExtModuleLaunchKernel (cat: cuda_runtime)


### Extracting GPU Kernels Directly

`get_gpu_events()` returns only the GPU kernels launched by a CPU op — skipping the intermediate runtime layer. This gives you a flat list of what actually executed on the GPU, along with each kernel's duration.

In [13]:
gpu_events = trace_tree.get_gpu_events(event_interest)
print(f"GPU kernels launched by {event_interest['name']} (total: {len(gpu_events)}):\n")
for gpu_evt in gpu_events: 
    print(f"  Kernel: {gpu_evt['name'][:60]}...")
    print(f"  Duration: {gpu_evt.get('dur', 0):.2f} µs\n")

GPU kernels launched by aten::convolution_backward (total: 4):

  Kernel: SubTensorOpWithScalar1d...
  Duration: 2.20 µs

  Kernel: igemm_bwd_gtcx3_nhwc_fp16_bx0_ex0_bt128x128x32_wt32x32x8_ws1...
  Duration: 16.30 µs

  Kernel: SubTensorOpWithScalar1d...
  Duration: 2.72 µs

  Kernel: igemm_wrw_gtcx3_nhwc_fp16_bx0_ex0_bt256x256x32_wt32x32x8_ws2...
  Duration: 15.14 µs



### Forward ↔ Backward Linking

TraceLens can link forward operations to their corresponding backward operation. Here we find an `aten::convolution` (forward) and use `include_bwd=True` to show its linked backward op in the subtree.

In [3]:
event_interest = next(
    evt
    for evt in trace_tree.events
    if evt.get("name") == "aten::convolution" and evt.get("cat") == "cpu_op"
)

print(f"Found operation: {event_interest['name']}")
print(f"Duration: {event_interest.get('dur', 0):.2f} µs")
print(f"UID: {event_interest['UID']}")

print(f"Subtree for {event_interest['name']} with backward ops:\n")
trace_tree.traverse_subtree_and_print(event_interest,include_bwd=True)

Found operation: aten::convolution
Duration: 104.95 µs
UID: 997
Subtree for aten::convolution with backward ops:

└── UID: 997, Category: cpu_op, Name: aten::convolution
    ├── UID: 999, Category: cpu_op, Name: aten::_convolution
    │   └── UID: 1000, Category: cpu_op, Name: aten::miopen_convolution
    │       └── UID: 22386, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
    │           └── UID: 23666, Category: kernel, Name: miopenSp3AsmConv_v30_3_1_gfx9_fp16_dot2_edc_f3x2_stride2, Duration: 49.433
    └── [BWD] autograd::engine::evaluate_function: ConvolutionBackward0 (UID: 15346)
        └── UID: 15347, Category: cpu_op, Name: ConvolutionBackward0
            └── UID: 15349, Category: cpu_op, Name: aten::convolution_backward
                ├── UID: 26634, Category: cuda_runtime, Name: hipExtModuleLaunchKernel
                │   └── UID: 27684, Category: kernel, Name: SubTensorOpWithScalar1d, Duration: 5.929
                ├── UID: 26636, Category: cuda_runtime, Name: 

---
## Performance Analysis with TreePerfAnalyzer

Now that we've seen how to navigate the tree structure, let's use `TreePerfAnalyzer` to get high-level performance insights. These APIs produce the same DataFrames that back the Excel performance report — but here you can interact with them programmatically.

In [4]:
from pprint import pprint
import json
import pandas as pd
from TraceLens.PerfModel import dict_cat2names

### GPU Timeline Breakdown

`get_df_gpu_timeline()` shows how GPU time is split between computation, communication, memory copies, and idle time — accounting for overlaps between categories. This is the same data as the `gpu_timeline` sheet in the Excel report.

In [5]:
# get breakdown of gpu timeline - busy time, idle time, communication time, etc
analyzer.get_df_gpu_timeline()

,type,time ms,percent
0,computation_time,47.634433,34.014377
1,exposed_comm_time,41.067746,29.325296
2,exposed_memcpy_time,2.487088,1.775958
3,busy_time,91.189267,65.115631
4,idle_time,48.852787,34.884369
5,total_time,140.042054,100.000000
6,total_comm_time,64.471378,46.037156
7,total_memcpy_time,3.396753,2.425523


### Kernel Launchers

`get_df_kernel_launchers()` returns the leaf-level CPU operations that actually launch GPU kernels, along with the GPU time each one induces. This avoids double counting — only the lowest-level ops in the call stack are included.

In [6]:
# table of all lowest-level CPU operations (from the call stack perspective)
# and the time they "induce" on the GPU
df_kernel_launchers = analyzer.get_df_kernel_launchers(include_kernel_details=True)
df_kernel_launchers.round(2).head()

,name,op category,UID,total_direct_kernel_time,total_subtree_kernel_time,direct_kernel_count,overlapping_kernel_names,overlapping_kernels_details,overlap_pct,Input Dims,Input type,Input Strides,Concrete Inputs,kernel_details,num_kernels,process_name,process_label,thread_name
0,aten::_local_scalar_dense,other,2,15.75,15.75,1,None,None,NaN,"((),)","(float,)","((),)","(,)","[{'name': 'Memcpy DtoH (Device -> Host)', 'dur...",1,python3,CPU,thread 3194582 (python3)
1,aten::mul,elementwise,3,4.73,4.73,1,None,None,NaN,"((), ())","(float, long int)","((), ())","(, )",[{'name': 'void at::native::vectorized_element...,1,python3,CPU,thread 3194582 (python3)
2,aten::_local_scalar_dense,other,5,14.65,14.65,1,None,None,NaN,"((),)","(float,)","((),)","(,)","[{'name': 'Memcpy DtoH (Device -> Host)', 'dur...",1,python3,CPU,thread 3194582 (python3)
3,aten::copy_,elementwise,11,60.45,60.45,1,None,None,NaN,"((256, 3, 32, 32), (256, 3, 32, 32), ())","(float, float, Scalar)","((3072, 1024, 32, 1), (3072, 1024, 32, 1), ())","(, , False)","[{'name': 'Memcpy HtoD (Host -> Device)', 'dur...",1,python3,CPU,thread 3194582 (python3)
4,aten::copy_,elementwise,15,5.73,5.73,1,None,None,NaN,"((256,), (256,), ())","(long int, long int, Scalar)","((1,), (1,), ())","(, , False)","[{'name': 'Memcpy HtoD (Host -> Device)', 'dur...",1,python3,CPU,thread 3194582 (python3)


In [7]:
# group by op name and summarize
# this gives an op wise breakdown of gpu time
df_kernel_launchers_summary = analyzer.get_df_kernel_launchers_summary(
    df_kernel_launchers
)
df_kernel_launchers_summary.round(2).head()

,name,total_direct_kernel_time_sum,total_subtree_kernel_time_sum,total_direct_kernel_time_ms,total_subtree_kernel_time_ms,Count,Categories,Percentage (%),Cumulative Percentage (%)
0,aten::convolution_backward,27006.06,27006.06,27.01,27.01,159,{CONV_bwd},52.92,52.92
1,aten::miopen_convolution,7732.87,7732.87,7.73,7.73,159,{CONV_fwd},15.15,68.07
2,aten::copy_,4723.57,4723.57,4.72,4.72,1320,{elementwise},9.26,77.33
3,aten::miopen_batch_norm,2175.47,2175.47,2.18,2.18,159,{NORM_fwd},4.26,81.59
4,aten::mul,1906.27,1906.27,1.91,1.91,492,{elementwise},3.74,85.33


### Unique Argument Combinations

`get_df_kernel_launchers_unique_args()` groups operations by their unique signature — (op name + input dims + types + strides + concrete args). This is the most detailed view, matching the `ops_unique_args` sheet, and lets you pinpoint which specific input shapes are slow. You can also filter to a single operation name to drill into its variants.

In [8]:
# unique argument combinations across all kernel-launching ops
analyzer.get_df_kernel_launchers_unique_args(df_kernel_launchers, include_pct=True)

,name,op category,process_name,process_label,thread_name,Input Dims,Input type,Input Strides,Concrete Inputs,operation_count,total_direct_kernel_time_mean,total_subtree_kernel_time_mean,total_direct_kernel_time_sum,total_subtree_kernel_time_sum,ex_UID,kernel_details_summary,Percentage (%),Cumulative Percentage (%)
0,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 256, 8, 8), (256, 64, 8, 8), (256, 64, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((16384, 64, 8, 1), (4096, 64, 8, 1), (64, 1, ...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",12,564.186910,564.186910,6770.242920,6770.242920,14878,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_MT64x128x32_M...,13.266865,13.266865
1,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 512, 4, 4), (256, 128, 4, 4), (512, 128...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((8192, 16, 4, 1), (2048, 16, 4, 1), (128, 1, ...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",12,306.317871,306.317871,3675.814453,3675.814453,14268,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,7.203070,20.469935
2,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 64, 8, 8), (256, 256, 8, 8), (64, 256, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((4096, 64, 8, 1), (16384, 64, 8, 1), (256, 1,...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",6,584.467244,584.467244,3506.803467,3506.803467,14970,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_MT64x128x32_M...,6.871879,27.341814
3,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 1024, 2, 2), (256, 1024, 1, 1), (), (),...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((4096, 4, 2, 1), (1024, 1, 1, 1), (), (), (),...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",15,167.320345,167.320345,2509.805176,2509.805176,1738,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,4.918176,32.259990
4,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 2048, 1, 1), (256, 1024, 2, 2), (2048, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((2048, 1, 1, 1), (4096, 4, 2, 1), (1024, 1, 1...","(, , , [0], [2, 2], [0, 0], [1, 1], False, [0,...",3,677.644694,677.644694,2032.934082,2032.934082,13184,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",3.983707,36.243696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249,aten::copy_,elementwise,python3,CPU,thread 3194582 (python3),"((64, 64, 1, 1), (64, 64, 1, 1), ())","(c10::Half, float, Scalar)","((64, 1, 1, 1), (64, 1, 1, 1), ())","(, , False)",3,2.483724,2.483724,7.451172,7.451172,1024,[{'name': 'void at::native::elementwise_kernel...,0.014601,99.948026
250,aten::copy_,elementwise,python3,CPU,thread 3194582 (python3),"((100,), (100,), ())","(c10::Half, float, Scalar)","((1,), (1,), ())","(, , False)",3,2.417318,2.417318,7.251953,7.251953,2402,[{'name': 'void at::native::elementwise_kernel...,0.014211,99.962236
251,aten::fill_,elementwise,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 100), ())","(float, Scalar)","((100, 1), ())","(, 0)",3,2.390706,2.390706,7.172119,7.172119,12792,[{'name': 'void at::native::vectorized_element...,0.014054,99.976291
252,aten::fill_,elementwise,python3,CPU,thread 3194582 (python3),"((), ())","(float, Scalar)","((), ())","(, 1.)",3,2.323324,2.323324,6.969971,6.969971,2450,[{'name': 'void at::native::vectorized_element...,0.013658,99.989949


In [10]:
# same as above, but restricted to a specific op type — e.g., only aten::convolution
# useful for drilling into the breakdown of a single op
analyzer.get_df_kernel_launchers_unique_args(
    df_kernel_launchers, event_name="aten::miopen_convolution", include_pct=True
)

,name,op category,process_name,process_label,thread_name,Input Dims,Input type,Input Strides,Concrete Inputs,operation_count,total_direct_kernel_time_mean,total_subtree_kernel_time_mean,total_direct_kernel_time_sum,total_subtree_kernel_time_sum,ex_UID,kernel_details_summary,Percentage (%),Cumulative Percentage (%)
0,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 1024, 2, 2), (256, 1024, 1, 1), (), (),...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((4096, 4, 2, 1), (1024, 1, 1, 1), (), (), (),...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",15,167.320345,167.320345,2509.805176,2509.805176,1738,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,32.456332,32.456332
1,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 512, 2, 2), (512, 512, 3, 3), (), (), (...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((2048, 4, 2, 1), (4608, 9, 3, 1), (), (), (),...","(, , , [1, 1], [2, 2], [1, 1], 1, True, False)",3,277.211589,277.211589,831.634766,831.634766,2164,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,10.754545,43.210877
2,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 256, 2, 2), (1024, 256, 1, 1), (), (), ...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((1024, 4, 2, 1), (256, 1, 1, 1), (), (), (), ...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",18,28.876180,28.876180,519.771240,519.771240,1687,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",6.721585,49.932462
3,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 256, 2, 2), (256, 256, 3, 3), (), (), (...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((1024, 4, 2, 1), (2304, 9, 3, 1), (), (), (),...","(, , , [1, 1], [1, 1], [1, 1], 1, True, False)",15,33.000911,33.000911,495.013672,495.013672,1764,"[{'name': 'batched_transpose_128x4_half', 'str...",6.401424,56.333886
4,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 64, 8, 8), (256, 64, 1, 1), (), (), (),...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((4096, 64, 8, 1), (64, 1, 1, 1), (), (), (), ...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",12,38.013123,38.013123,456.157471,456.157471,1081,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,5.898943,62.232829
5,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 512, 4, 4), (1024, 512, 1, 1), (), (), ...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((8192, 16, 4, 1), (512, 1, 1, 1), (), (), (),...","(, , , [0, 0], [2, 2], [1, 1], 1, True, False)",3,124.491374,124.491374,373.474121,373.474121,1710,[{'name': 'Cijk_Ailk_Bljk_HHS_BH_MT64x64x64_MI...,4.829698,67.062527
6,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 1024, 2, 2), (2048, 1024, 1, 1), (), ()...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((4096, 4, 2, 1), (1024, 1, 1, 1), (), (), (),...","(, , , [0, 0], [2, 2], [1, 1], 1, True, False)",3,117.534668,117.534668,352.604004,352.604004,2213,[{'name': 'Cijk_Ailk_Bljk_HHS_BH_MT64x32x64_MI...,4.559809,71.622336
7,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 256, 8, 8), (64, 256, 1, 1), (), (), ()...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((16384, 64, 8, 1), (256, 1, 1, 1), (), (), ()...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",6,50.735067,50.735067,304.410400,304.410400,1132,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,3.936578,75.558914
8,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 128, 4, 4), (128, 128, 3, 3), (), (), (...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((2048, 16, 4, 1), (1152, 9, 3, 1), (), (), ()...","(, , , [1, 1], [1, 1], [1, 1], 1, True, False)",9,30.565348,30.565348,275.088135,275.088135,1421,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,3.557388,79.116303
9,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 64, 8, 8), (64, 64,

---
## Roofline / Performance Metrics

TraceLens can compute roofline-style performance metrics for supported op categories: **GEMM**, **CONV** (fwd+bwd), **SDPA** (Flash Attention), **Unary/Binary Elementwise**, and more. For each operation, it calculates theoretical GFLOPS, data moved (MB), arithmetic intensity (FLOPS/Byte), and achieved TFLOPS/s and TB/s.

We use `dict_cat2names` to get the op names for each category, then `compute_perf_metrics()`, `build_df_perf_metrics()`, and `summarize_df_perf_metrics()` to analyze them.

### Example 1: GEMM Ops

Find all GEMM (matrix multiply) events in the trace — `aten::addmm`, `aten::mm`, `aten::bmm`, etc. We inspect a single event's raw perf metrics, then build and summarize a table across all GEMM occurrences.

In [11]:
# Find all GEMM events in the trace
gemm_op_names = dict_cat2names["GEMM"]
gemm_events = [
    event for event in analyzer.tree.events if event["name"] in gemm_op_names
]
print(f"Found {len(gemm_events)} GEMM events")

if gemm_events:
    # take an example event and compute perf metrics
    gemm_event = gemm_events[0]
    print("\nEvent dict:")
    pprint(gemm_event)
    print("\nPerf metrics dict:")
    pprint(analyzer.compute_perf_metrics(gemm_event))

Found 9 GEMM events

Event dict:
{<TraceKeys.UID: 'UID'>: 2413,
 'args': {'Concrete Inputs': ['', '', '', '1', '1'],
          'Ev Idx': 2183,
          'External id': 2184,
          'Fwd thread id': 0,
          'Input Dims': [[100], [256, 2048], [2048, 100], [], []],
          'Input Strides': [[1], [2048, 1], [1, 2048], [], []],
          'Input type': ['c10::Half',
                         'c10::Half',
                         'c10::Half',
                         'Scalar',
                         'Scalar'],
          'Record function id': 0,
          'Sequence number': 2345},
 'bwd_events': [12804],
 'cat': 'cpu_op',
 'children': [2415, 23310, 23312, 23314, 23316],
 'direct_kernel_count': 2,
 'dur': 116.933,
 'gpu_events': [24454, 24456],
 'kernel_details': [{'dur': 3.805,
                     'name': 'void at::native::elementwise_kernel<128, 4, '
                             'at::native::gpu_kernel_impl_nocast<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lamb

In [12]:
# build table of compute perf metrics for all GEMM events
# include_kernel_details=True adds a column with the kernel names launched by each CPU op
if gemm_events:
    df_gemm_ops = analyzer.build_df_perf_metrics(
        gemm_events, include_kernel_details=True
    )
    display(df_gemm_ops.head())
else:
    print("No GEMM events found in this trace")

,cat,name,UID,pid,tid,process_name,process_label,thread_name,external_id,overlapping_kernel_names,...,param: N,param: K,param: bias,param: stride_A,param: stride_B,param: dtype_A_B,param: B,param: transpose,kernel_details,num_kernels
0,cpu_op,aten::addmm,2413,3194582,3194582,python3,CPU,thread 3194582 (python3),2184,None,...,100,2048,True,"(2048, 1)","(1, 2048)","(c10::Half, c10::Half)",1,"(True, False)",[{'name': 'void at::native::elementwise_kernel...,2
1,cpu_op,aten::addmm,6674,3194582,3194582,python3,CPU,thread 3194582 (python3),9282,None,...,100,2048,True,"(2048, 1)","(1, 2048)","(c10::Half, c10::Half)",1,"(True, False)",[{'name': 'void at::native::elementwise_kernel...,2
2,cpu_op,aten::addmm,10935,3194582,3194582,python3,CPU,thread 3194582 (python3),16380,None,...,100,2048,True,"(2048, 1)","(1, 2048)","(c10::Half, c10::Half)",1,"(True, False)",[{'name': 'void at::native::elementwise_kernel...,2
3,cpu_op,aten::mm,12810,3194582,3196065,python3,CPU,thread 3196065 (pt_autograd_0),2583,None,...,2048,100,False,"(100, 1)","(2048, 1)","(c10::Half, c10::Half)",1,"(False, False)",[{'name': 'Cijk_Ailk_Bljk_HHS_BH_Bias_AS_SAV_U...,1
4,cpu_op,aten::mm,12814,3194582,3196065,python3,CPU,thread 3196065 (pt_autograd_0),2587,None,...,2048,256,False,"(1, 100)","(2048, 1)","(c10::Half, c10::Half)",1,"(False, True)",[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_Bias_AS_SAV_U...,1


In [18]:
# summarize by grouping across params M, K, N and bias and computing aggregate metrics
if gemm_events:
    display(analyzer.summarize_df_perf_metrics(df_gemm_ops, ["mean"]))
else:
    print("No GEMM events found in this trace")

,name,param: M,param: N,param: K,param: bias,param: stride_A,param: stride_B,param: dtype_A_B,param: B,param: transpose,...,TFLOPS/s_mean,process_name_first,process_label_first,thread_name_first,Compute Spec,kernel_details__summarize_kernel_stats,Kernel Time (µs)_mean,Kernel Time (µs)_sum,name_count,UID_first
0,aten::addmm,256,100,2048,True,"(2048, 1)","(1, 2048)","('c10::Half', 'c10::Half')",1,"(True, False)",...,7.589106,python3,CPU,thread 3194582 (python3),matrix_fp16,[{'name': 'void at::native::elementwise_kernel...,13.820231,41.460693,3,2413
1,aten::mm,256,2048,100,False,"(100, 1)","(2048, 1)","('c10::Half', 'c10::Half')",1,"(False, False)",...,13.826637,python3,CPU,thread 3196065 (pt_autograd_0),matrix_fp16,[{'name': 'Cijk_Ailk_Bljk_HHS_BH_Bias_AS_SAV_U...,7.584635,22.753906,3,12810
2,aten::mm,100,2048,256,False,"(1, 100)","(2048, 1)","('c10::Half', 'c10::Half')",1,"(False, True)",...,15.015173,python3,CPU,thread 3196065 (pt_autograd_0),matrix_fp16,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_Bias_AS_SAV_U...,6.983643,20.950928,3,12814


### Example 2: Unary Elementwise Ops

Perf metrics for unary elementwise operations — `aten::relu`, `aten::sigmoid`, `aten::tanh`, etc. These are typically memory-bound (low arithmetic intensity), so the key metric to watch is TB/s rather than TFLOPS/s.

In [20]:
# Example 4: Unary elementwise
unary_elemwise_op_names = dict_cat2names["UnaryElementwise"]
unary_elementwise_events = [
    event
    for event in analyzer.tree.events
    if event["name"] in unary_elemwise_op_names
]
print(f"Found {len(unary_elementwise_events)} unary elementwise events")

if unary_elementwise_events:
    df_unary_elementwise_ops = analyzer.build_df_perf_metrics(unary_elementwise_events)
    display(analyzer.summarize_df_perf_metrics(df_unary_elementwise_ops, ["mean"]))
else:
    print("No unary elementwise events found in this trace")

Found 1467 unary elementwise events


,name,param: op_shape,param: dtype_in_out,param: stride_input,param: stride_output,GFLOPS_first,Data Moved (MB)_first,FLOPS/Byte_first,TB/s_mean,TFLOPS/s_mean,process_name_first,process_label_first,thread_name_first,Compute Spec,Kernel Time (µs)_mean,Kernel Time (µs)_sum,name_count,UID_first
0,aten::copy_,"(256,)","('float', 'float')","(1,)","(1,)",2.560000e-07,0.001953,0.125000,0.000952,1.189669e-04,python3,CPU,thread 3194582 (python3),vector_fp32,2.897616,556.342285,192,615
1,aten::copy_,(),"('long int', None)",(),None,1.000000e-09,0.000015,0.062500,0.000009,5.884667e-07,python3,CPU,thread 3194582 (python3),,2.890837,459.643066,159,933
2,aten::copy_,"(512,)","('float', 'float')","(1,)","(1,)",5.120000e-07,0.003906,0.125000,0.002048,2.560019e-04,python3,CPU,thread 3194582 (python3),vector_fp32,2.775335,366.344238,132,635
3,aten::copy_,"(64,)","('float', 'float')","(1,)","(1,)",6.400000e-08,0.000488,0.125000,0.000207,2.586539e-05,python3,CPU,thread 3194582 (python3),vector_fp32,3.328700,279.610840,84,613
4,aten::copy_,"(128,)","('float', 'float')","(1,)","(1,)",1.280000e-07,0.000977,0.125000,0.000484,6.048791e-05,python3,CPU,thread 3194582 (python3),vector_fp32,2.767054,265.637207,96,637
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88,aten::copy_,"(256,)","('float', 'bool')","(1,)","(1,)",2.560000e-07,NaN,NaN,NaN,9.782583e-05,python3,CPU,thread 3194582 (python3),vector_fp32,2.617025,7.851074,3,2440
89,aten::copy_,"(100,)","('float', 'c10::Half')","(1,)","(1,)",1.000000e-07,0.000572,0.166667,0.000236,3.930878e-05,python3,CPU,thread 3196065 (pt_autograd_0),vector_fp32,2.550700,7.652100,3,12845
90,aten::clamp_min,"(256, 512, 2, 2)","('c10::Half', None)","(2048, 4, 2, 1)",None,5.242880e-04,2.000000,0.250000,0.828008,2.070020e-01,python3,CPU,thread 3194582 (python3),vector_fp16,2.537028,7.611084,3,2153
91,aten::copy_,"(64, 64, 1, 1)","('c10::Half', 'float')","(64, 1, 1, 1)","(64, 1, 1, 1)",4.096000e-06,0.023438,0.166667,0.009900,1.649984e-03,python3,CPU,thread 3194582 (python3),vector_fp16,2.483724,7.451172,3,1024
